In [0]:
%pip install gspread oauth2client pandas

In [0]:
%pip install xgboost

In [0]:
# Importing Libraries

from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col
import xgboost as xgb
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.metrics import (
    classification_report, roc_auc_score,
    average_precision_score, confusion_matrix, f1_score, precision_recall_curve
)
from sklearn.metrics import confusion_matrix as cm_fn
from mlflow.models.signature import infer_signature
import os

# Volume and catalog setup
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.ml_layer.mlflow_tmp")
print("Volume created")

os.environ['MLFLOW_DFS_TMP'] = '/Volumes/workspace/ml_layer/mlflow_tmp'

def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()

app_train   = spark.table("workspace.ml_layer.application_train_features")
app_test    = spark.table("workspace.ml_layer.application_test_features")
trans_train = spark.table("workspace.ml_layer.transaction_train_features")
trans_test  = spark.table("workspace.ml_layer.transaction_test_features")

# Optuna Integration

def load_best_params(dataset_name, model_name="XGBoost"):
    """
    Load Optuna-optimized params from Delta. Falls back to manual if missing.
    Returns None if Optuna hasn't been run yet.
    """
    try:
        params_df = spark.table("workspace.ml_layer.best_hyperparameters") \
            .filter(f"dataset = '{dataset_name}' AND model = '{model_name}'") \
            .toPandas()

        if len(params_df) == 0:
            return None

        best_params = dict(zip(params_df["parameter"], params_df["value"]))

        # Cast integer params back to int (Delta stores all as double)
        int_params = ["n_estimators", "max_depth", "min_child_weight"]
        for p in int_params:
            if p in best_params:
                best_params[p] = int(best_params[p])

        print(f" Loaded Optuna best params for {dataset_name}")
        return best_params

    except Exception as e:
        print(f"  ⚠️  Optuna params not found for {dataset_name} ({type(e).__name__})")
        return None

# Evaluation Functions

def evaluate_spark_model(predictions, model_name, dataset_name):
    evaluator_roc = BinaryClassificationEvaluator(
        labelCol="is_fraud",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    evaluator_pr = BinaryClassificationEvaluator(
        labelCol="is_fraud",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR"
    )

    roc_auc = evaluator_roc.evaluate(predictions)
    pr_auc  = evaluator_pr.evaluate(predictions)

    pred_pd = predictions.select(
        col("is_fraud").cast("double"),
        vector_to_array("probability").getItem(1).alias("fraud_prob")
    ).toPandas()

    y_true  = pred_pd["is_fraud"].values
    y_proba = pred_pd["fraud_prob"].values

    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores      = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_idx       = f1_scores.argmax()
    best_threshold = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.5
    best_f1        = float(f1_scores[best_idx])

    y_pred_optimal = (y_proba >= best_threshold).astype(int)
    tn, fp, fn, tp = cm_fn(y_true, y_pred_optimal).ravel()

    print(f"\n{'='*50}")
    print(f"  {model_name} | {dataset_name}")
    print(f"{'='*50}")
    print(f"  ROC-AUC        : {roc_auc:.4f}")
    print(f"  PR-AUC         : {pr_auc:.4f}   ← key metric for fraud")
    print(f"  Best threshold : {best_threshold:.4f}")
    print(f"  F1 @ threshold : {best_f1:.4f}")
    print(f"\n{classification_report(y_true, y_pred_optimal, target_names=['legit', 'fraud'])}")

    return {
        "roc_auc"         : roc_auc,
        "pr_auc"          : pr_auc,
        "f1"              : best_f1,
        "best_threshold"  : best_threshold,
        "true_positives"  : int(tp),
        "false_positives" : int(fp),
        "false_negatives" : int(fn),
        "true_negatives"  : int(tn)
    }


def evaluate_sklearn_model(y_true, y_pred, y_prob, model_name, dataset_name):
    roc_auc = roc_auc_score(y_true, y_prob)
    pr_auc  = average_precision_score(y_true, y_prob)

    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores      = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_idx       = f1_scores.argmax()
    best_threshold = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.5
    best_f1        = float(f1_scores[best_idx])

    y_pred_optimal = (y_prob >= best_threshold).astype(int)
    tn, fp, fn, tp = cm_fn(y_true, y_pred_optimal).ravel()

    print(f"\n{'='*50}")
    print(f"  {model_name} | {dataset_name}")
    print(f"{'='*50}")
    print(f"  ROC-AUC        : {roc_auc:.4f}")
    print(f"  PR-AUC         : {pr_auc:.4f}   ← key metric for fraud")
    print(f"  Best threshold : {best_threshold:.4f}")
    print(f"  F1 @ threshold : {best_f1:.4f}")
    print(f"\n{classification_report(y_true, y_pred_optimal, target_names=['legit', 'fraud'])}")
    print(confusion_matrix(y_true, y_pred_optimal))

    return {
        "roc_auc"         : roc_auc,
        "pr_auc"          : pr_auc,
        "f1"              : best_f1,
        "best_threshold"  : best_threshold,
        "true_positives"  : int(tp),
        "false_positives" : int(fp),
        "false_negatives" : int(fn),
        "true_negatives"  : int(tn)
    }


# Sanity Check

def check_dataset(df, name):
    fraud_type  = dict(df.dtypes)["is_fraud"]
    fraud_count = df.filter(col("is_fraud") == 1).count()
    total       = df.count()
    print(f"{name}: is_fraud type={fraud_type} | fraud={fraud_count:,} / {total:,} "
          f"({100*fraud_count/total:.2f}%)")

check_dataset(app_train,   "app_train  ")
check_dataset(app_test,    "app_test   ")
check_dataset(trans_train, "trans_train")
check_dataset(trans_test,  "trans_test ")


# Logistic Regression

def run_logistic_regression(train_df, test_df, dataset_name):
    ensure_catalog()
    train_df = train_df.withColumn("is_fraud", col("is_fraud").cast("double"))
    test_df  = test_df.withColumn("is_fraud", col("is_fraud").cast("double"))

    lr = LogisticRegression(
        featuresCol="features",
        labelCol="is_fraud",
        maxIter=100,
        regParam=0.01,
        elasticNetParam=0,
        threshold=0.5
    )
    with mlflow.start_run(run_name=f"LR_{dataset_name}") as run:
        model       = lr.fit(train_df)
        predictions = model.transform(test_df)
        metrics     = evaluate_spark_model(predictions, "Logistic Regression", dataset_name)

        mlflow.log_metrics(metrics)
        mlflow.log_param("dataset", dataset_name)
        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_param("hyperparameter_source", "Manual")

        signature = infer_signature(
            model_input  = train_df.select("features").limit(5).toPandas(),
            model_output = predictions.select("prediction").limit(5).toPandas()
        )
        mlflow.spark.log_model(
            spark_model           = model,
            artifact_path         = "lr_model",
            signature             = signature,
            registered_model_name = f"Modelo_Fraude_LR_{dataset_name.capitalize()}"
        )
        print(f"LR model registered | Run ID: {run.info.run_id}")

    return model, predictions


# Random Forest

def run_random_forest(train_df, test_df, dataset_name):
    ensure_catalog()
    train_df = train_df.withColumn("is_fraud", col("is_fraud").cast("double"))
    test_df  = test_df.withColumn("is_fraud", col("is_fraud").cast("double"))

    if dataset_name == "transactions":
        max_bins         = 256
        num_trees        = 60
        max_depth        = 7
        subsampling_rate = 0.8
        min_instances    = 20
        min_info_gain    = 0.001
    else:
        max_bins         = 256
        num_trees        = 100
        max_depth        = 10
        subsampling_rate = 1.0
        min_instances    = 5
        min_info_gain    = 0.0

    rf = RandomForestClassifier(
        featuresCol           = "features",
        labelCol              = "is_fraud",
        numTrees              = num_trees,
        maxDepth              = max_depth,
        maxBins               = max_bins,
        minInstancesPerNode   = min_instances,
        subsamplingRate       = subsampling_rate,
        minInfoGain           = min_info_gain,
        impurity              = "gini",
        featureSubsetStrategy = "sqrt",
        seed                  = 42
    )
    with mlflow.start_run(run_name=f"RF_{dataset_name}") as run:
        model       = rf.fit(train_df)
        predictions = model.transform(test_df)
        metrics     = evaluate_spark_model(predictions, "Random Forest", dataset_name)

        mlflow.log_metrics(metrics)
        mlflow.log_param("dataset", dataset_name)
        mlflow.log_param("num_trees", num_trees)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("hyperparameter_source", "Manual")

        print("\n  Feature Importances:")
        for i, score in enumerate(model.featureImportances):
            print(f"    Feature {i}: {score:.4f}")

        signature = infer_signature(
            model_input  = train_df.select("features").limit(5).toPandas(),
            model_output = predictions.select("prediction").limit(5).toPandas()
        )

        mlflow.spark.log_model(
            spark_model           = model,
            artifact_path         = "rf_model",
            signature             = signature,
            registered_model_name = f"Modelo_Fraude_RF_{dataset_name.capitalize()}"
        )
        print(f"RF model registered | Run ID: {run.info.run_id}")

    return model, predictions


# XGBOOST — auto-loads Optuna params

def run_xgboost(train_df, test_df, dataset_name):
    ensure_catalog()
    print(f"  Converting {dataset_name} to pandas...")

    train_arr = train_df.withColumn("features_arr", vector_to_array("features"))
    test_arr  = test_df.withColumn("features_arr", vector_to_array("features"))

    X_train = np.array(train_arr.select("features_arr").toPandas()["features_arr"].tolist())
    y_train = train_arr.select("is_fraud").toPandas()["is_fraud"].values
    X_test  = np.array(test_arr.select("features_arr").toPandas()["features_arr"].tolist())
    y_test  = test_arr.select("is_fraud").toPandas()["is_fraud"].values

    fraud_count      = int(y_train.sum())
    legit_count      = len(y_train) - fraud_count
    scale_pos_weight = legit_count / fraud_count
    print(f"  scale_pos_weight: {scale_pos_weight:.1f} ({legit_count} legit / {fraud_count} fraud)")

    # Try Optuna params first, fall back to manual 
    best_params = load_best_params(dataset_name, "XGBoost")

    if best_params:
        # Use Optuna-optimized hyperparameters
        xgb_model = xgb.XGBClassifier(
            **best_params,
            scale_pos_weight      = scale_pos_weight,
            eval_metric           = "aucpr",
            early_stopping_rounds = 40,
            random_state          = 42,
            n_jobs                = -1
        )
        param_source = "Optuna-optimized"

    else:
        # Fallback to manual hyperparameters
        if dataset_name == "transactions":
            xgb_model = xgb.XGBClassifier(
                n_estimators          = 400,
                max_depth             = 5,
                learning_rate         = 0.03,
                subsample             = 0.75,
                colsample_bytree      = 0.7,
                colsample_bylevel     = 0.8,
                min_child_weight      = 20,
                gamma                 = 0.1,
                reg_alpha             = 0.05,
                reg_lambda            = 1.5,
                scale_pos_weight      = scale_pos_weight,
                eval_metric           = "aucpr",
                early_stopping_rounds = 40,
                random_state          = 42,
                n_jobs                = -1
            )
        else:
            xgb_model = xgb.XGBClassifier(
                n_estimators          = 500,
                max_depth             = 4,
                learning_rate         = 0.02,
                subsample             = 0.7,
                colsample_bytree      = 0.6,
                colsample_bylevel     = 0.7,
                min_child_weight      = 15,
                gamma                 = 0.2,
                reg_alpha             = 0.1,
                reg_lambda            = 2.0,
                scale_pos_weight      = scale_pos_weight,
                eval_metric           = "aucpr",
                early_stopping_rounds = 50,
                random_state          = 42,
                n_jobs                = -1
            )
        param_source = "Manual"

    print(f"  Hyperparameter source: {param_source}")

    with mlflow.start_run(run_name=f"XGB_{dataset_name}") as run:
        # Track parameter provenance — used by dashboard for Optuna vs Manual comparison
        mlflow.log_param("hyperparameter_source", param_source)

        xgb_model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=50
        )
        y_prob = xgb_model.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)

        metrics = evaluate_sklearn_model(y_test, y_pred, y_prob, "XGBoost", dataset_name)
        mlflow.log_metrics(metrics)
        mlflow.log_param("dataset", dataset_name)
        mlflow.log_param("scale_pos_weight", round(scale_pos_weight, 2))

        # Log all hyperparameters dynamically — works for both manual and Optuna
        params_to_log = [
            "n_estimators", "max_depth", "learning_rate",
            "subsample", "colsample_bytree", "colsample_bylevel",
            "min_child_weight", "gamma", "reg_alpha", "reg_lambda"
        ]
        for p in params_to_log:
            value = xgb_model.get_params().get(p)
            if value is not None:
                try:
                    mlflow.log_param(p, value)
                except Exception:
                    pass

        signature = infer_signature(
            model_input  = X_train[:5],
            model_output = xgb_model.predict_proba(X_train[:5])
        )

        mlflow.xgboost.log_model(
            xgb_model             = xgb_model,
            artifact_path         = "xgb_model",
            signature             = signature,
            registered_model_name = f"Modelo_Fraude_XGB_{dataset_name.capitalize()}"
        )
        print(f"  XGB model registered | Run ID: {run.info.run_id}")

    return xgb_model, y_prob, y_test

# Running all models

username = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{username}/supervised_models")
print(f"MLflow experiment: /Users/{username}/supervised_models")

# Applications
lr_app,  lr_app_preds              = run_logistic_regression(app_train, app_test, "applications")
rf_app,  rf_app_preds              = run_random_forest(app_train, app_test, "applications")
xgb_app, xgb_app_prob, y_app_test = run_xgboost(app_train, app_test, "applications")

# Transactions
lr_trans,  lr_trans_preds               = run_logistic_regression(trans_train, trans_test, "transactions")
rf_trans,  rf_trans_preds               = run_random_forest(trans_train, trans_test, "transactions")
xgb_trans, xgb_trans_prob, y_trans_test = run_xgboost(trans_train, trans_test, "transactions")

In [0]:
spark.sql("SHOW CATALOGS").show()